# Part 3 Starter Notebook

This notebook is a **read-only analysis surface** for part 3.
Use it to inspect and visualize the outputs generated by the `query_eval` evaluation subsystem.

Use this notebook when you want to:
- load the latest `evaluation_results/query_eval/...` run
- inspect aggregate CSV outputs
- identify non-exact cells quickly
- make first-pass runtime, memory, and correctness plots

Do **not** use this notebook to re-implement the query engine.
If you need to run experiments, use:
- `python3 -m query_eval.cli run-suite ...`
- or `query_eval.runner.run_suite(...)`

If you need project context first, read:
- `query_eval/README.md`


## 1. Resolve a Results Directory

By default, this notebook loads the newest directory under `evaluation_results/query_eval/`.
If you want to analyze a specific run instead, set `RESULTS_DIR` explicitly in the next cell.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use('ggplot')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

RESULTS_ROOT = Path('evaluation_results/query_eval')
RESULTS_DIR = None  # Example: Path('evaluation_results/query_eval/20260415_021241_smoke_suite')


def resolve_results_dir(results_dir: Path | None = None) -> Path:
    if results_dir is not None:
        if not results_dir.exists():
            raise FileNotFoundError(f'Requested results directory does not exist: {results_dir}')
        return results_dir

    if not RESULTS_ROOT.exists():
        raise FileNotFoundError(
            'No evaluation results directory found. Run the suite first with '            '`python3 -m query_eval.cli run-suite ...`.'
        )

    run_dirs = sorted([path for path in RESULTS_ROOT.iterdir() if path.is_dir()])
    if not run_dirs:
        raise FileNotFoundError(
            'No run directories found under evaluation_results/query_eval/. '            'Run the suite first.'
        )
    return run_dirs[-1]


results_dir = resolve_results_dir(RESULTS_DIR)
print(f'Using results directory: {results_dir}')


## 2. Load the Run Artifacts

This cell loads:
- `manifest.json`
- `raw_runs.jsonl`
- `cell_level_aggregate.csv`
- `query_level_aggregate.csv`
- `dataset_level_aggregate.csv`
- `suite_summary.csv`


In [ ]:
manifest = json.loads((results_dir / 'manifest.json').read_text())
raw_df = pd.read_json(results_dir / 'raw_runs.jsonl', lines=True)
cell_df = pd.read_csv(results_dir / 'cell_level_aggregate.csv')
query_df = pd.read_csv(results_dir / 'query_level_aggregate.csv')
dataset_df = pd.read_csv(results_dir / 'dataset_level_aggregate.csv')
suite_df = pd.read_csv(results_dir / 'suite_summary.csv')

print('Run created at:', manifest['created_at'])
print('Code version:', manifest['code_version'])
print('Datasets:', ', '.join(manifest['datasets']))
print('Queries:', ', '.join(manifest['queries']))
print('Modes:', ', '.join(manifest['modes']))
print('Raw run records:', len(raw_df))
print('Cell rows:', len(cell_df))

print('\nRun config:')
display(pd.DataFrame([manifest['run_config']]))

print('\nCell-level aggregates:')
display(cell_df.head())


## 3. Quick Trust Checks

These tables answer the first questions a part-3 analyst usually has:
- which cells were non-exact?
- where did the optimization lose recall?
- are there any false positives or false negatives?


In [ ]:
non_exact_cells = cell_df.loc[~cell_df['all_runs_exact_set_match']].copy()
non_exact_cells = non_exact_cells.sort_values(
    by=['median_recall', 'median_f1', 'dataset_slug', 'query_id', 'mode_name'],
    ascending=[True, True, True, True, True],
)

print('Non-exact cells:', len(non_exact_cells))
display(
    non_exact_cells[
        [
            'dataset_slug',
            'query_id',
            'mode_name',
            'median_match_count',
            'median_fp',
            'median_fn',
            'median_precision',
            'median_recall',
            'median_f1',
            'all_runs_exact_set_match',
        ]
    ]
)


## 4. Suite-Level Performance View

This first-pass plot is for top-level comparison across the three modes.
It is useful for presentation slides and early narrative framing.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

suite_df.plot.bar(x='mode_name', y='median_wall_time_ms', legend=False, ax=axes[0], color='#4C78A8')
axes[0].set_title('Median Wall Time by Mode')
axes[0].set_ylabel('Wall Time (ms)')
axes[0].tick_params(axis='x', rotation=25)

suite_df.plot.bar(x='mode_name', y='median_cpu_time_ms', legend=False, ax=axes[1], color='#F58518')
axes[1].set_title('Median CPU Time by Mode')
axes[1].set_ylabel('CPU Time (ms)')
axes[1].tick_params(axis='x', rotation=25)

suite_df.plot.bar(x='mode_name', y='median_peak_rss_mb', legend=False, ax=axes[2], color='#54A24B')
axes[2].set_title('Median Peak RSS by Mode')
axes[2].set_ylabel('Peak RSS (MB)')
axes[2].tick_params(axis='x', rotation=25)

fig.suptitle('Suite Summary', fontsize=14)
fig.tight_layout()
plt.show()


## 5. Dataset-Level Runtime View

This plot helps answer whether some datasets are systematically harder than others.


In [ ]:
runtime_pivot = dataset_df.pivot(index='dataset_slug', columns='mode_name', values='median_wall_time_ms')
runtime_pivot = runtime_pivot.sort_index()

ax = runtime_pivot.plot(kind='bar', figsize=(12, 6))
ax.set_title('Dataset-Level Median Wall Time by Mode')
ax.set_xlabel('Dataset')
ax.set_ylabel('Wall Time (ms)')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()


## 6. Correctness View for the Optimization Candidate

This is often the most useful first-pass correctness figure for part 3.
It isolates `minor_optimization` so recall / F1 losses are obvious.


In [ ]:
minor_df = cell_df.loc[cell_df['mode_name'] == 'minor_optimization'].copy()
minor_df['dataset_query'] = minor_df['dataset_slug'] + ' | ' + minor_df['query_id']
minor_df = minor_df.sort_values(by=['dataset_slug', 'query_id'])

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

axes[0].bar(minor_df['dataset_query'], minor_df['median_recall'], color='#E45756')
axes[0].set_title('Minor Optimization Recall by Dataset / Query')
axes[0].set_ylabel('Recall')
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=70)

axes[1].bar(minor_df['dataset_query'], minor_df['median_f1'], color='#72B7B2')
axes[1].set_title('Minor Optimization F1 by Dataset / Query')
axes[1].set_ylabel('F1')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=70)

fig.tight_layout()
plt.show()


## 7. Raw Repetition Variability

Aggregate tables are useful, but part 3 may also want to discuss spread across repetitions.
This cell extracts measured run timings from the raw JSONL ledger and plots them by mode.


In [ ]:
measured_raw_df = raw_df.loc[~raw_df['is_warmup']].copy()
measured_raw_df['wall_time_ms'] = measured_raw_df['timing'].apply(lambda item: item['wall_time_ms'])
measured_raw_df['cpu_time_ms'] = measured_raw_df['timing'].apply(lambda item: item['cpu_time_ms'])
measured_raw_df['peak_rss_mb'] = measured_raw_df['memory'].apply(lambda item: item['peak_rss_mb'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

measured_raw_df.boxplot(column='wall_time_ms', by='mode_name', ax=axes[0], grid=False)
axes[0].set_title('Wall Time Distribution by Mode')
axes[0].set_xlabel('Mode')
axes[0].set_ylabel('Wall Time (ms)')
axes[0].tick_params(axis='x', rotation=25)

measured_raw_df.boxplot(column='cpu_time_ms', by='mode_name', ax=axes[1], grid=False)
axes[1].set_title('CPU Time Distribution by Mode')
axes[1].set_xlabel('Mode')
axes[1].set_ylabel('CPU Time (ms)')
axes[1].tick_params(axis='x', rotation=25)

measured_raw_df.boxplot(column='peak_rss_mb', by='mode_name', ax=axes[2], grid=False)
axes[2].set_title('Peak RSS Distribution by Mode')
axes[2].set_xlabel('Mode')
axes[2].set_ylabel('Peak RSS (MB)')
axes[2].tick_params(axis='x', rotation=25)

plt.suptitle('')
fig.tight_layout()
plt.show()


## 8. Suggested Next Steps for Part 3

Good next analysis moves from here:
- inspect `non_exact_cells` in more detail
- compare `full_decompression` and `minor_optimization` directly on wall time and recall
- read `raw_runs.jsonl` for sampled false positives / false negatives
- build final presentation plots from `cell_df`, `dataset_df`, and `suite_df`
- avoid re-implementing the query engine in notebook cells
